In [1]:
# import libraries
import pandas as pd
import geopandas as gpd
import ee
import geemap

## Connect to Google Earth Engine (GEE)

In [2]:
# Authenticate GEE
ee.Authenticate()

# Initialize GEE
EE_PROJECT_ID = ""   # Change to your project ID

# ee.Initialize(project=EE_PROJECT_ID)
ee.Initialize()


Successfully saved authorization token.


## Visualization Parameters

In [9]:
# Center coordinates to show map
kano_center =  (11.999986,  8.551571)

In [10]:
# Boundary visualization params 
vis_params_fao_1 = {
  "fillColor": 'b5ffb4',
  "color": '00909F',
  "width": 1.0,
}

vis_params_aoi = {"fillcolor": "", "color": "red"}

# Sentinel-2 Visualization parameters
vis_params_s2_rgb = {
    'min': 300,
    'max': 3000,
    'bands': ['B4', 'B3', 'B2'],
}

# DEM visualization params


# Precipitation vis params


## Get Boundary Data

In [11]:
# Download admin boundaries
# Reference link for data: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_SIMPLIFIED_500m_2015_level0

fao_gaul_l0 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level0') # Country boundaries
fao_gaul_l1 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level1') # State boundaries
fao_gaul_l2 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level2') # LGA boundaries

# Creat map to visualize the FAO GAUL boundary data
boundary_map = geemap.Map(center=(6.7, 7.5), zoom=8)

boundary_map.addLayer(fao_gaul_l0, {}, 'Country Boundaries')
boundary_map.addLayer(fao_gaul_l1, {}, 'State Boundaries')
boundary_map.addLayer(fao_gaul_l2, {}, 'LGA Boundaries')
boundary_map

Map(center=[6.7, 7.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tr…

In [12]:
# Check columns
print(fao_gaul_l0.limit(0).getInfo()["columns"])

# Extract Nigerian boundaries from FAO GAUL
nga_l0 = fao_gaul_l0.filter(ee.Filter.eq("ADM0_NAME", "Nigeria")) # Single Nigeria boundary
nga_l1 = fao_gaul_l1.filter(ee.Filter.eq("ADM0_NAME", "Nigeria")) # State boundaries in Nigeria
nga_l2 = fao_gaul_l2.filter(ee.Filter.eq("ADM0_NAME", "Nigeria")) # LGAs boundaries in Nigeria

kano_l0 = nga_l1.filter(ee.Filter.eq("ADM1_NAME", "Kano")) # Kano boundary/extent
aky_lga = nga_l2.filter(ee.Filter.eq("ADM2_NAME", "Akinyele")) # Akinyele LGA in Ibadan

print(kano_l0.getInfo()) #
print(aky_lga.getInfo())

# Get geometry from Kano boundary (FeatureCollection)
aoi = kano_l0.geometry()
aoi_bbox = aoi.bounds()

# Creat map to visualize Kano boundary data
aoi_map = geemap.Map(center=kano_center, zoom=10)
aoi_map.addLayer(kano_l0, vis_params_aoi, 'Kano Boundary')
aoi_map.addLayer(aky_lga, vis_params_aoi, 'Akinyele LGA')
aoi_map

{'ADM0_CODE': 'Integer', 'ADM0_NAME': 'String', 'DISP_AREA': 'String', 'EXP0_YEAR': 'Integer', 'STATUS': 'String', 'STR0_YEAR': 'Integer', 'Shape_Area': 'Float', 'Shape_Leng': 'Float', 'system:index': 'String'}
{'type': 'FeatureCollection', 'columns': {'ADM0_CODE': 'Integer', 'ADM0_NAME': 'String', 'ADM1_CODE': 'Integer', 'ADM1_NAME': 'String', 'DISP_AREA': 'String', 'EXP1_YEAR': 'Integer', 'STATUS': 'String', 'STR1_YEAR': 'Integer', 'Shape_Area': 'Float', 'Shape_Leng': 'Float', 'system:index': 'String'}, 'version': 1701682755394127, 'id': 'FAO/GAUL_SIMPLIFIED_500m/2015/level1', 'properties': {'system:asset_size': 80042928}, 'features': [{'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[9.225564587198297, 11.343312217065831], [9.227810346981562, 11.352295212685663], [9.223318805404384, 11.379244292092107], [9.23005608473727, 11.40619333700369], [9.241284865279248, 11.437633943701458], [9.252513600108383, 11.453354210946427], [9.286199934329725, 11.484794822766556], 

Map(center=[11.999986, 8.551571], controls=(WidgetControl(options=['position', 'transparent_bg'], position='to…

## Explore Image operations

In [18]:
# Explore Sentinel-2 image collection
s2_img_col = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') # All S2 images for the entire world
    .filterDate('2020-01-01', '2020-01-30') # Limite to specific period/date 
    .filterBounds(kano_l0.geometry()) # Limit search to Kano
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)) # Cloud percentage < 30% 
)

# Check number of images in 's2_img_col'
print(f"Number of images in S2 collection: {s2_img_col.size().getInfo()}\n")

# Check properties of the 's2_img_col'
print(s2_img_col.getInfo())

# Take only the first image in 's2_img_col'
first_s2_img = s2_img_col.first()
s2_img_bands = first_s2_img.bandNames().getInfo()

print(f"First image in S2 collection: {first_s2_img.getInfo()}\n")
print(f"Bands in S2 images: {s2_img_bands}\n")


# Select only RGB bands
first_s2_img_rgb = first_s2_img.select(["B2", "B4"])
print(f"Bands in S2 images [RGB]: {first_s2_img_rgb.bandNames().getInfo()}\n")


# Create a map to visualise S2 image & boundary data
s2_map = geemap.Map(center=kano_center, zoom=6)
s2_map.add_basemap("SATELLITE")

# Add layers to map
s2_map.addLayer(kano_l0, vis_params_aoi, "Kano Boundary")
s2_map.addLayer(first_s2_img.clip(kano_l0.geometry()), vis_params_s2_rgb, "S2 First Image")
s2_map

Number of images in S2 collection: 51

{'type': 'ImageCollection', 'bands': [], 'version': 1784647305389500.0, 'id': 'COPERNICUS/S2_SR_HARMONIZED', 'properties': {'date_range': [1490659200000, 1647907200000], 'period': 0, 'system:visualization_0_min': '0.0', 'type_name': 'ImageCollection', 'keywords': ['copernicus', 'esa', 'eu', 'msi', 'reflectance', 'sentinel', 'sr'], 'system:visualization_0_bands': 'B4,B3,B2', 'thumb': 'https://mw1.google.com/ges/dd/images/COPERNICUS_S2_SR_thumb.png', 'description': '<p>Sentinel-2 is a wide-swath, high-resolution, multi-spectral\nimaging mission supporting Copernicus Land Monitoring studies,\nincluding the monitoring of vegetation, soil and water cover,\nas well as observation of inland waterways and coastal areas.</p><p>The Sentinel-2 L2 data are downloaded from scihub. They were\ncomputed by running sen2cor. WARNING: ESA did not produce L2 data\nfor all L1 assets, and earlier L2 coverage is not global.</p><p>The assets contain\n12 UINT16 spectral b

Map(center=[11.999986, 8.551571], controls=(WidgetControl(options=['position', 'transparent_bg'], position='to…

In [19]:
# Mosaic the entire S2 collection & clip to Kano extent [Spatial Mosaic]
s2_mosaic = s2_img_col.mosaic()
s2_mosaic_clipped = s2_mosaic.clip(kano_l0.geometry())
print(f"Mosaiced S2 Collection {s2_mosaic.getInfo()}")

s2_map.addLayer(s2_mosaic_clipped, vis_params_s2_rgb, "S2 Mosaiced - Spatial")
s2_map

Mosaiced S2 Collection {'type': 'Image', 'bands': [{'id': 'B1', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B2', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B3', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B4', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B5', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B6', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B7', 'data_type': {'type': 'PixelTy

Map(center=[11.999986, 8.551571], controls=(WidgetControl(options=['position', 'transparent_bg'], position='to…

In [20]:
# Compute median mosaic the entire S2 collection & clip to Kano extent [Temporal Mosaic]
s2_median = s2_img_col.median().clip(kano_l0.geometry())
print(f"Mosaiced S2 Collection {s2_mosaic.getInfo()}")

s2_map.addLayer(s2_median, vis_params_s2_rgb, "S2 Median Mosaiced")
s2_map

Mosaiced S2 Collection {'type': 'Image', 'bands': [{'id': 'B1', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B2', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B3', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B4', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B5', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B6', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B7', 'data_type': {'type': 'PixelTy

Map(bottom=7942.0, center=[11.999986, 8.551571], controls=(WidgetControl(options=['position', 'transparent_bg'…

# . Thematic / Groundwater-Influencing Layers

## Digital ELevation Model (DEM) / Slope

In [21]:
# 